In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip uninstall -y monai monai-weekly >/dev/null 2>&1

!pip install -q \
    "monai>=1.4.0" \
    nibabel \
    h5py \
    pyyaml \
    tensorboardX \
    scikit-image \
    scipy \
    tqdm \
    wandb

import torch, monai
print("✓ PyTorch:", torch.__version__)
print("✓ CUDA:   ", torch.version.cuda)
print("✓ MONAI:  ", monai.__version__)

In [ ]:
%cd /content

print("\n" + "="*70)
print("CLONING RCPS REPOSITORY FROM GITHUB")
print("="*70 + "\n")

# Remove any existing RCPS folder
!rm -rf RCPS

# Clone the repository
print("Cloning from: https://github.com/hsiangyuzhao/RCPS.git\n")
!git clone https://github.com/hsiangyuzhao/RCPS.git

print("\n" + "="*70)
print("✓ Repository cloned successfully!")
print("="*70)

print("\nRCPS structure:")
!ls -la /content/RCPS

# Set RCPS_ROOT for later use
RCPS_ROOT = "/content/RCPS"
print(f"\nRCPS_ROOT: {RCPS_ROOT}")

In [ ]:
import os
import random

print("="*70)
print("CREATING TRAIN/TEST SPLITS")
print("="*70 + "\n")

# Paths
RCPS_ROOT = "/content/RCPS"
DRIVE_LA_TRAIN = "/content/drive/MyDrive/Datasets/LA/2018LA_Seg_Training Set"

# Get all patient IDs from your Drive
all_patients = sorted([d for d in os.listdir(DRIVE_LA_TRAIN) 
                       if os.path.isdir(os.path.join(DRIVE_LA_TRAIN, d))])

print(f"✓ Found {len(all_patients)} patients")
print(f"  First: {all_patients[0]}")
print(f"  Last:  {all_patients[-1]}\n")

# Create 80/20 split (80 train, 20 test)
# Use fixed seed for reproducibility
random.seed(1337)
random.shuffle(all_patients)

split_idx = 80  # First 80 for training
train_patients = sorted(all_patients[:split_idx])
test_patients = sorted(all_patients[split_idx:])

print(f"Split: {len(train_patients)} train / {len(test_patients)} test\n")

# Create data directory
LA_LIST_DIR = f"{RCPS_ROOT}/data/LA"
os.makedirs(LA_LIST_DIR, exist_ok=True)

# Save train.list
TRAIN_LIST = f"{LA_LIST_DIR}/train.list"
with open(TRAIN_LIST, 'w') as f:
    f.write('\n'.join(train_patients) + '\n')

print(f"✓ Created: {TRAIN_LIST}")
print(f"  First 5: {train_patients[:5]}")

# Save test.list
TEST_LIST = f"{LA_LIST_DIR}/test.list"
with open(TEST_LIST, 'w') as f:
    f.write('\n'.join(test_patients) + '\n')

print(f"\n✓ Created: {TEST_LIST}")
print(f"  First 5: {test_patients[:5]}")

print("\n" + "="*70)
print("✅ Dataset splits created successfully!")
print("="*70)

In [ ]:
import shutil

print("\n" + "="*70)
print("COPYING LA DATASET")
print("="*70 + "\n")

DRIVE_LA_TRAIN = "/content/drive/MyDrive/Datasets/LA/2018LA_Seg_Training Set"
DEST_LA_TRAIN = f"{LA_LIST_DIR}/2018LA_Seg_Training Set"

# Remove old copy
if os.path.exists(DEST_LA_TRAIN):
    shutil.rmtree(DEST_LA_TRAIN)

# Copy dataset
print(f"Copying from: {DRIVE_LA_TRAIN}")
print(f"To: {DEST_LA_TRAIN}\n")

shutil.copytree(DRIVE_LA_TRAIN, DEST_LA_TRAIN)

print("\n✓ Dataset copied!")
print(f"\nContents (first 10):")
contents = os.listdir(DEST_LA_TRAIN)[:10]
for item in contents:
    print(f"  {item}")

In [ ]:
print("\n" + "="*70)
print("VERIFYING DATASET SPLITS")
print("="*70 + "\n")

print("First 10 lines of train.list:")
with open(TRAIN_LIST) as f:
    for i, line in enumerate(f):
        if i >= 10:
            break
        print(f"  {i+1}. {line.strip()}")

print("\nFirst 10 lines of test.list:")
with open(TEST_LIST) as f:
    for i, line in enumerate(f):
        if i >= 10:
            break
        print(f"  {i+1}. {line.strip()}")

# Count total
with open(TRAIN_LIST) as f:
    train_count = len([l for l in f if l.strip()])
with open(TEST_LIST) as f:
    test_count = len([l for l in f if l.strip()])

print(f"\n✓ Total: {train_count} train + {test_count} test = {train_count + test_count} patients")

In [ ]:
import os
import h5py
import nibabel as nib
import numpy as np
from tqdm import tqdm

print("\n" + "="*70)
print("PREPROCESSING LA DATASET FOR RCPS")
print("="*70 + "\n")

# Paths
LA_H5_ROOT = f"{LA_LIST_DIR}/2018LA_Seg_Training Set"
RCPS_LA_ROOT = f"{RCPS_ROOT}/data/LA_rcps"

# Clean and create output directories
!rm -rf "$RCPS_LA_ROOT"
for sub in ["train_images", "train_labels", "val_images", "val_labels"]:
    os.makedirs(os.path.join(RCPS_LA_ROOT, sub), exist_ok=True)

def convert_split(list_path, img_dest, lbl_dest):
    """
    Convert H5 files to NIfTI format for RCPS
    """
    print(f"\nProcessing {os.path.basename(list_path)}...")
    
    with open(list_path) as f:
        patient_ids = [line.strip() for line in f if line.strip()]
    
    print(f"  Total patients: {len(patient_ids)}")
    
    for patient_id in tqdm(patient_ids, desc="Converting"):
        h5_path = os.path.join(LA_H5_ROOT, patient_id, "mri_norm2.h5")
        
        if not os.path.exists(h5_path):
            print(f"  ⚠️  Missing: {patient_id}")
            continue
        
        try:
            # Load H5 file
            with h5py.File(h5_path, 'r') as f:
                image = f['image'][:].astype(np.float32)
                label = f['label'][:].astype(np.uint8)
            
            # Save as NIfTI
            img_nii = nib.Nifti1Image(image, affine=np.eye(4))
            lbl_nii = nib.Nifti1Image(label, affine=np.eye(4))
            
            nib.save(img_nii, os.path.join(img_dest, f"{patient_id}.nii.gz"))
            nib.save(lbl_nii, os.path.join(lbl_dest, f"{patient_id}.nii.gz"))
            
        except Exception as e:
            print(f"  ❌ Error with {patient_id}: {e}")
            continue
    
    print(f"  ✓ Processed {len(patient_ids)} patients")

# Convert training set
convert_split(
    TRAIN_LIST,
    os.path.join(RCPS_LA_ROOT, "train_images"),
    os.path.join(RCPS_LA_ROOT, "train_labels")
)

# Convert test/validation set
convert_split(
    TEST_LIST,
    os.path.join(RCPS_LA_ROOT, "val_images"),
    os.path.join(RCPS_LA_ROOT, "val_labels")
)

print("\n" + "="*70)
print("✅ PREPROCESSING COMPLETE")
print("="*70)

print("\nDataset structure:")
!ls -lh "$RCPS_LA_ROOT"

print("\nSample counts:")
for sub in ["train_images", "train_labels", "val_images", "val_labels"]:
    count = len(os.listdir(os.path.join(RCPS_LA_ROOT, sub)))
    print(f"  {sub}: {count} files")

In [ ]:
%cd /content/RCPS

print("\n" + "="*70)
print("STARTING RCPS TRAINING")
print("="*70)
print("Configuration:")
print("  Task: LA (Left Atrium)")
print("  Method: RCPS (Rectified Contrastive Pseudo Supervision)")
print("  Labeled: 10% (~8 patients)")
print("  Epochs: 400")
print("  GPU: A100")
print("="*70 + "\n")

!python train.py \
    --task la \
    --exp_name la_colab_test \
    --data_root ./data/LA_rcps \
    --train_list ./data/LA/train.list \
    --val_list ./data/LA/test.list \
    --max_epoch 400 \
    --batch_size 4 \
    --base_lr 0.01 \
    --labeled_ratio 0.1 \
    --seed 1337

In [ ]:
# Compress results for download
!cd /content/RCPS/experiments && \
  tar -czf results.tar.gz checkpoints/ logs/

from google.colab import files
print("\n📦 Downloading results...")
files.download('/content/RCPS/experiments/results.tar.gz')

In [ ]:
# Compress predictions for download
import shutil

EXP_NAME = "la_colab_test-task_la-ratio_0.1"  # Update if your exp_name differs

PRED_DIR = f"/content/RCPS/experiments/inference_display/la/{EXP_NAME}"

if os.path.exists(PRED_DIR):
    shutil.make_archive('/content/predictions', 'zip', PRED_DIR)
    print("✓ Predictions compressed")
    
    from google.colab import files
    files.download('/content/predictions.zip')
else:
    print(f"❌ Predictions not found at: {PRED_DIR}")
    print("\nAvailable experiments:")
    !ls /content/RCPS/experiments/inference_display/la/

In [ ]:
import os
import nibabel as nib
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider, Dropdown

print("="*70)
print("INTERACTIVE RESULTS VIEWER")
print("="*70 + "\n")

# Paths (update EXP_NAME if needed)
EXP_NAME = "la_colab_test-task_la-ratio_0.1"

IMAGE_DIR = "/content/RCPS/data/LA_rcps/val_images"
LABEL_DIR = "/content/RCPS/data/LA_rcps/val_labels"
PRED_DIR = f"/content/RCPS/experiments/inference_display/la/{EXP_NAME}"

# Get available cases
if os.path.exists(PRED_DIR):
    pred_files = sorted([f.replace('_pred.nii.gz', '') 
                        for f in os.listdir(PRED_DIR) 
                        if f.endswith('_pred.nii.gz')])
    print(f"✓ Found {len(pred_files)} predictions\n")
else:
    print(f"❌ Predictions not found. Run training first!")
    pred_files = []

if pred_files:
    def visualize_case(case_name, slice_idx=None):
        """Show: Original | Prediction | Ground Truth | Comparison"""
        
        # Load data
        image = nib.load(f"{IMAGE_DIR}/{case_name}.nii.gz").get_fdata()
        label = nib.load(f"{LABEL_DIR}/{case_name}.nii.gz").get_fdata()
        pred = nib.load(f"{PRED_DIR}/{case_name}_pred.nii.gz").get_fdata()
        
        # Default to middle slice
        if slice_idx is None:
            slice_idx = image.shape[2] // 2
        
        # Create figure
        fig, axes = plt.subplots(1, 4, figsize=(20, 5))
        
        # 1. Original
        axes[0].imshow(image[:, :, slice_idx].T, cmap='gray', origin='lower')
        axes[0].set_title(f'Original MRI\n{case_name[:20]}...', fontsize=14, fontweight='bold')
        axes[0].axis('off')
        
        # 2. Prediction
        axes[1].imshow(image[:, :, slice_idx].T, cmap='gray', origin='lower')
        mask = pred[:, :, slice_idx] > 0
        if mask.any():
            axes[1].imshow(mask.T, cmap='Reds', origin='lower', alpha=0.5)
        axes[1].set_title('Prediction (Red)', fontsize=14, fontweight='bold')
        axes[1].axis('off')
        
        # 3. Ground Truth
        axes[2].imshow(image[:, :, slice_idx].T, cmap='gray', origin='lower')
        mask_gt = label[:, :, slice_idx] > 0
        if mask_gt.any():
            axes[2].imshow(mask_gt.T, cmap='Greens', origin='lower', alpha=0.5)
        axes[2].set_title('Ground Truth (Green)', fontsize=14, fontweight='bold')
        axes[2].axis('off')
        
        # 4. Comparison
        axes[3].imshow(image[:, :, slice_idx].T, cmap='gray', origin='lower', alpha=0.7)
        if mask.any():
            axes[3].imshow(mask.T, cmap='Reds', origin='lower', alpha=0.4)
        if mask_gt.any():
            axes[3].imshow(mask_gt.T, cmap='Greens', origin='lower', alpha=0.4)
        axes[3].set_title('Comparison\n(Red=Pred, Green=GT)', fontsize=14, fontweight='bold')
        axes[3].axis('off')
        
        plt.tight_layout()
        plt.show()
        
        # Calculate Dice
        intersection = ((pred > 0) & (label > 0)).sum()
        dice = 2.0 * intersection / ((pred > 0).sum() + (label > 0).sum() + 1e-8)
        
        print(f"\n📊 Metrics for {case_name}")
        print(f"  Slice: {slice_idx}/{image.shape[2]}")
        print(f"  3D Dice Score: {dice:.4f}")
    
    # Interactive viewer
    first_case = pred_files[0]
    image = nib.load(f"{IMAGE_DIR}/{first_case}.nii.gz").get_fdata()
    max_slice = image.shape[2] - 1
    
    interact(
        visualize_case,
        case_name=Dropdown(options=pred_files, description='Case:'),
        slice_idx=IntSlider(min=0, max=max_slice, value=max_slice//2, 
                           description='Slice:', continuous_update=False)
    )

In [ ]:
import torch

print("="*70)
print("MODEL CHECKPOINT INFO")
print("="*70 + "\n")

EXP_NAME = "la_colab_test-task_la-ratio_0.1"
CKPT_PATH = f"/content/RCPS/experiments/checkpoints/la/{EXP_NAME}/latest.pt"

if os.path.exists(CKPT_PATH):
    ckpt = torch.load(CKPT_PATH, map_location="cpu")
    
    print(f"Checkpoint: {CKPT_PATH}\n")
    print("Contents:")
    
    if isinstance(ckpt, dict):
        for key in ckpt.keys():
            print(f"  - {key}")
        
        if 'epoch' in ckpt:
            print(f"\n✓ Trained Epochs: {ckpt['epoch']}")
        if 'best_metric' in ckpt:
            print(f"✓ Best Metric: {ckpt['best_metric']:.4f}")
        if 'best_metric_epoch' in ckpt:
            print(f"✓ Best Epoch: {ckpt['best_metric_epoch']}")
    else:
        print(f"  Type: {type(ckpt)}")
else:
    print(f"❌ Checkpoint not found: {CKPT_PATH}")